In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

from itertools import islice

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools import ToolContext

from google.genai import types
from typing import Optional,Dict,Any


from google.adk.agents import LlmAgent, LoopAgent
from neo4j_for_adk import tool_success, tool_error, graphdb

from helper import make_agent_caller

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported")

Libraries imported
Libraries imported


In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = os.getenv("DEEPSEEK_MODEL")
llm = LiteLlm(model=MODEL_NAME)

print(llm.llm_client.completion(
    model=llm.model,
    messages=[{"role": "user", "content": "你准备好了吗？"}],
    tools=[],
))

print("\n Deepseek已经准备好了")

ModelResponse(id='c7067405-ec08-4cbf-9a34-3879a98860ec', created=1788774779, model='deepseek-v4-flash', object='chat.completion', system_fingerprint='a26a7955944dc5c60445bff77fac9c8e', choices=[Choices(finish_reason='stop', index=0, message=Message(content='准备好了！有什么我可以帮你的，请随时告诉我。', role='assistant', tool_calls=None, function_call=None, reasoning_content='我们需要回答用户“你准备好了吗？”中文。需要准备什么？可能用户要开始提问。我们作为助手应友好回应，表示准备好了。注意当前无上下文。可以简单回复。', provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=51, prompt_tokens=87, total_tokens=138, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=39, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=87))

 Deepseek已经准备好了

In [3]:
neo4j_is_ready = graphdb.send_query("RETURN 'Neo4j is Ready!' as message")

print(neo4j_is_ready)

{'status': 'success', 'query_result': [{'message': 'Neo4j is Ready!'}]}


In [4]:
## 7.3. Named Entity Recognition (NER) Sub-agent

In [5]:
# 命名实体识别 (NER) 智能体的角色和目标
ner_agent_role_and_goal = """
  你是一个顶级的算法程序，专用于分析文本文件。
  你的目标是：根据用户的具体需求，建议可以从文本中提取哪些相关的“命名实体”（Named Entities）。
  """

In [6]:
# 命名实体识别 (NER) 智能体的提示与规则
ner_agent_hints = """
  实体（Entities）指的是人、地点、事物和属性/性质，但不包括数量（quantities）。
  你的目标是提出一个实体【类型】（type）的列表，而不是提出实体的具体【实例】（instances）。

  在识别实体类型时，通常有两种主要方法：
  - 已知实体（well-known entities）：这些实体类型与现有图谱模式（graph schema）中已设定的“节点标签”紧密对应。
  - 发现实体（discovered entities）：这些实体类型可能还不存在于图谱模式中，但它们在源文本中频繁且稳定地出现。

  已知实体（well-known entities）的设计规则：
  - 始终使用现有的已知实体类型。例如，如果图谱中已经有一个已知类型叫 "Person"（人），而文本中也提到了人，那么请直接建议使用 "Person" 作为实体类型。
  - 优先复用（reusing）现有的实体类型，而不是去创建新的。

  发现实体（discovered entities）的设计规则：
  - 发现实体必须是在文本中被反复提及，并且与用户的最终目标高度相关的。
  - 始终去寻找那些能为现有图谱增加“深度”或“广度”的实体。
  - 例如，如果用户的目标是描绘社交网络，并且当前图谱已经有了 "Person"（人）节点，那么你应该通读文本，去发现像 "Hobby"（爱好）或 "Event"（事件）这样有价值的新实体类型。
  - 避免将数量/数值类型作为实体。数值类型最好作为现有实体或关系上的“属性（property）”来表示。
  - 例如，不要把 "Age"（年龄）建议为一个实体类型。年龄更适合作为 "Person" 实体上的一个附加属性 "age" 来表示。
"""

In [7]:
# 命名实体识别 (NER) 智能体的思维链指导步骤
ner_agent_chain_of_thought_directions = """
  任务准备阶段：
  - 使用 'get_user_goal'（获取用户目标）工具，来了解用户的最终目标是什么
  - 使用 'get_approved_files'（获取授权文件）工具，来获取你可以处理的文件列表
  - 使用 'get_well_known_types'（获取已知类型）工具，来获取当前图谱中已批准的节点标签（即现有分类）

  一步一步地思考（Think step by step）：
  1. 使用 'sample_file'（抽样文件）工具，对部分文件进行抽样阅读，以理解文本的内容。
  2. 思考文本中提到了哪些“已知实体”（well-known entities，即已存在于图谱中的类型）。
  3. 发掘文本中频繁出现、且有助于实现用户目标的“发现实体”（discovered entities，即需要新增的类型）。
  4. 使用 'set_proposed_entities'（设置建议实体）工具，将你找出的（已知的和新发现的）实体类型列表保存下来。
  5. 使用 'get_proposed_entities'（获取建议实体）工具，把你刚才保存的建议实体提取出来，并展示给用户看，请求用户批准（approval）。
  6. 如果用户批准了，使用 'approve_proposed_entities'（批准建议实体）工具，来最终确认这些实体类型。
  7. 如果用户不批准，请仔细考虑他们的反馈意见，并对你的建议列表进行迭代（重新修改并提交）。
"""

In [8]:
ner_agent_instruction = f"""
{ner_agent_role_and_goal}
{ner_agent_hints}
{ner_agent_chain_of_thought_directions}
"""

In [9]:
# 用于提议（propose）和批准（approve）实体类型的工具（常量定义，作为字典的键）
PROPOSED_ENTITIES = "proposed_entity_types" # 键名：建议的实体类型
APPROVED_ENTITIES = "approved_entity_types" # 键名：已批准的实体类型

def set_proposed_entities(proposed_entity_types: list[str], tool_context:ToolContext) -> dict:
    """设置准备从非结构化文本中提取的“建议实体类型”列表。"""
    # 将智能体生成的列表保存到 tool_context（工具上下文/状态）中
    tool_context.state[PROPOSED_ENTITIES] = proposed_entity_types
    # 返回成功状态给智能体
    return tool_success(PROPOSED_ENTITIES, proposed_entity_types)

def get_proposed_entities(tool_context:ToolContext) -> dict:
    """获取准备从非结构化文本中提取的“建议实体类型”列表。"""
    # 从状态字典中读取之前保存的建议列表，如果没有则返回空列表 []
    return tool_context.state.get(PROPOSED_ENTITIES, [])

def approve_proposed_entities(tool_context:ToolContext) -> dict:
    """在获得用户批准后，将“建议的实体类型”记录为“已批准的实体类型”列表。

    【注意】：仅当用户明确批准了建议的文件/类型时，才可调用此工具。
    """
    # 错误处理：如果状态里根本就没有“建议列表”，说明智能体还没做第一步，报错。
    if PROPOSED_ENTITIES not in tool_context.state:
        return tool_error("没有可批准的建议实体类型。请先设置建议实体，请求用户批准，然后再调用此工具。")

    # 核心逻辑：把“建议列表”里的数据，复制给“已批准列表”
    tool_context.state[APPROVED_ENTITIES] = tool_context.state.get(PROPOSED_ENTITIES)
    # 返回成功状态给智能体
    return tool_success(APPROVED_ENTITIES, tool_context.state[APPROVED_ENTITIES])

def get_approved_entities(tool_context:ToolContext) -> dict:
    """获取最终批准的、将从非结构化文本中提取的“实体类型”列表。"""
    # 从状态字典中读取最终批准的列表
    return tool_context.state.get(APPROVED_ENTITIES, [])

In [10]:
def get_well_known_types(tool_context:ToolContext) -> dict:
    """获取代表图谱模式（graph schema）中“已知实体类型（well-known entity types）”的、已批准的节点标签（labels）。"""

    # 从智能体的上下文状态（记忆）中，读取“已批准的图谱构建计划（approved_construction_plan）”
    # 如果状态里没有这个计划，就默认返回一个空字典 {}
    construction_plan = tool_context.state.get("approved_construction_plan", {})

    # Python 的集合推导式 (Set Comprehension)
    # 逻辑：遍历构建计划中的每一个条目（entry）
    # 如果该条目的构建类型（construction_type）是 "node"（节点/实体）
    # 就把它的 "label"（标签名称，比如 "Person", "Company"）提取出来，并去重
    approved_labels = {entry["label"] for entry in construction_plan.values() if entry["construction_type"] == "node"}

    # 向大模型返回工具调用成功的信息，并附带提取出来的已有分类标签
    return tool_success("approved_labels", approved_labels)

In [16]:
from tools import get_approved_user_goal, get_approved_files, sample_file
ner_agent_tools = [
    get_approved_user_goal, get_approved_files, sample_file,
    get_well_known_types,
    set_proposed_entities,
    get_proposed_entities,
    approve_proposed_entities
]

In [12]:
file_result = sample_file("product_reviews/gothenburg_table_reviews.md")

print(file_result["content"])

# Gothenburg Table Reviews

Scraped from https://www.between2furns.com/Gothenburg-Table/dp/B0BQJWJWJW

## Rating: ★★★★★ (5/5)
This table is the centerpiece of our dining room! The modern design is stunning yet timeless, and the quality is exceptional. We've had it for 6 months now and it still looks brand new despite daily use with two kids. Assembly took about an hour with two people. The surface is easy to clean and surprisingly resistant to scratches. Highly recommend!

- @akollegger (Cambridge)

---

## Rating: ★★★☆☆ (3/5)
The Gothenburg Table looks beautiful, I'll give it that. But assembly was a nightmare - some of the pre-drilled holes didn't line up properly and I had to drill new ones. Once assembled, it's sturdy enough, but I expected better quality control for this price point. The table top also shows fingerprints very easily which is annoying for daily use.

- @furniture_lover92 (Seattle)

---

## Rating: ★★★★☆ (4/5)
love love LOVE the look of this table!! perfect size for

In [13]:
# 设定 NER 智能体的名称常量（加了 v1 版本号，方便后续迭代）
NER_AGENT_NAME = "ner_schema_agent_v1"

# 实例化（创建）一个智能体 (Agent) 对象
ner_schema_agent = Agent(
    name=NER_AGENT_NAME,
    # 智能体的名称，系统和其他智能体会通过这个名字来识别它

    description="提出可以从文本文件中提取的命名实体类型（的建议）。",
    # 智能体的描述。在多智能体系统中，其他 Agent 看到了这个描述，就知道有相关任务可以交给它来做

    model=llm,
    # 智能体的“大脑”：指定驱动这个 Agent 的大语言模型 (如 GPT-4、Claude 等实例)

    instruction=ner_agent_instruction,
    # 智能体的“灵魂/人设”：把我们前面定义的角色、目标、规则(hints)和思维链(CoT)打包传给它

    tools=ner_agent_tools,
    # 智能体的“手脚”：传入之前定义好的 get_well_known_types 等工具列表，让它有能力操作外部系统
)

In [14]:
# 命名实体识别 (NER) 智能体的初始状态（即它的初始“短期记忆”）
ner_agent_initial_state = {

    # 1. 已批准的用户目标 (对应 CoT 里的 get_user_goal 工具)
    "approved_user_goal": {
        "kind_of_graph": "供应链分析", # 用户想要构建的图谱类型
        "description": """用于制造产品的多层级物料清单（BOM），可用于根本原因分析。
        请加入产品评论数据，以便从（用户）报告的问题（如质量、组装难度或耐用性）开始进行分析。"""
    },

    # 2. 已批准处理的文件列表 (对应 CoT 里的 get_approved_files 工具)
    "approved_files": [
        "product_reviews/gothenburg_table_reviews.md",     # 哥德堡_桌子_评论
        "product_reviews/helsingborg_dresser_reviews.md",  # 赫尔辛堡_梳妆台_评论
        "product_reviews/jonkoping_coffee_table_reviews.md", # 延雪平_咖啡桌_评论
        "product_reviews/linkoping_bed_reviews.md",        # 林雪平_床_评论
        "product_reviews/malmo_desk_reviews.md",           # 马尔默_书桌_评论
        "product_reviews/norrkoping_nightstand_reviews.md",# 诺尔雪平_床头柜_评论
        "product_reviews/orebro_lamp_reviews.md",          # 厄勒布鲁_灯具_评论
        "product_reviews/stockholm_chair_reviews.md",      # 斯德哥尔摩_椅子_评论
        "product_reviews/uppsala_sofa_reviews.md",         # 乌普萨拉_沙发_评论
        "product_reviews/vasteras_bookshelf_reviews.md"    # 韦斯特罗斯_书架_评论
    ],

    # 3. 已批准的图谱构建计划 (对应 CoT 里的 get_well_known_types 工具！)
    "approved_construction_plan": {
        "Product": {                      # 这是外层的 Key
            "construction_type": "node",  # 构建类型：节点（实体）
            "label": "Product",           # 标签名：产品
        },
        "Assembly": {
            "construction_type": "node",
            "label": "Assembly",          # 标签名：组件/组装件
        },
        "Part": {
            "construction_type": "node",
            "label": "Part",              # 标签名：零件
        },
        "Supplier": {
            "construction_type": "node",
            "label": "Supplier",          # 标签名：供应商
        }
        # 英文原注释：在这里省略了“关系（Relationship）”的构建计划，因为在这个 Jupyter Notebook 教程中不会用到它。
    }
}

In [17]:
# 从辅助工具包中导入 make_agent_caller（这通常是课程提供的一个封装函数，用于管理智能体会话）
from helper import make_agent_caller

# 将之前创建好的智能体 (ner_schema_agent) 和初始状态记忆 (ner_agent_initial_state) 绑定，
# 生成一个可以直接调用的执行器对象。这里使用了 await，说明这是一个异步操作。
ner_agent_caller = await make_agent_caller(ner_schema_agent, initial_state=ner_agent_initial_state)

# 🚀 见证奇迹的时刻：向智能体发送任务指令，唤醒并让它开始干活！
# 指令内容：“将产品评论添加到知识图谱中，以便通过制造流程追溯产品投诉的根本原因。”
await ner_agent_caller.chat("将产品评论添加到知识图谱中，以便通过制造流程追溯产品投诉的根本原因。")

# 另外，如果你想看到智能体“每一步在想什么”（详细日志），可以取消注释下面这行代码（传入 True 表示开启 verbose 详细模式）
# await ner_agent_caller.call("Add product reviews.", True)

# 智能体执行完毕后，获取当前的会话状态（也就是它工作完之后的“最终记忆”）
session_end = await ner_agent_caller.get_session()

print("\n---\n")

# 打印最终的系统状态字典
print("\nSession state: ", session_end.state)

# 检查智能体是否按要求生成了“建议的实体” (PROPOSED_ENTITIES)
if PROPOSED_ENTITIES in session_end.state:
    print("\n建议的实体列表: ", session_end.state[PROPOSED_ENTITIES])

# 🛑 这里是一个精妙的测试点！
# 检查状态中是否直接出现了“已批准的实体” (APPROVED_ENTITIES)
if APPROVED_ENTITIES in session_end.state:
    # 如果有，说明智能体【违规操作】了！它没有等人类点头，自己就偷偷把草稿转正了。
    print("\n违规批准的实体 : ", session_end.state[APPROVED_ENTITIES])
else:
    # 正常情况应该走这里：智能体乖乖地停留在“等待老板批准”的阶段。
    print("\n正在等待批准)")


>>>👤 用户: 将产品评论添加到知识图谱中，以便通过制造流程追溯产品投诉的根本原因。
<<< Agent Response: Now I present the proposed entity types to the user for approval.


---


Session state:  {'approved_user_goal': {'kind_of_graph': '供应链分析', 'description': '用于制造产品的多层级物料清单（BOM），可用于根本原因分析。\n        请加入产品评论数据，以便从（用户）报告的问题（如质量、组装难度或耐用性）开始进行分析。'}, 'approved_files': ['product_reviews/gothenburg_table_reviews.md', 'product_reviews/helsingborg_dresser_reviews.md', 'product_reviews/jonkoping_coffee_table_reviews.md', 'product_reviews/linkoping_bed_reviews.md', 'product_reviews/malmo_desk_reviews.md', 'product_reviews/norrkoping_nightstand_reviews.md', 'product_reviews/orebro_lamp_reviews.md', 'product_reviews/stockholm_chair_reviews.md', 'product_reviews/uppsala_sofa_reviews.md', 'product_reviews/vasteras_bookshelf_reviews.md'], 'approved_construction_plan': {'Product': {'construction_type': 'node', 'label': 'Product'}, 'Assembly': {'construction_type': 'node', 'label': 'Assembly'}, 'Part': {'construction_type': 'node', 'label': '

In [18]:
# 1. 人类介入：向智能体发送明确的“批准”指令。
# 听到这句话后，智能体终于满足了提示词中“只有用户明确批准后才能调用”的条件。
await ner_agent_caller.chat("批准这些建议的实体。") # 中文意为：“批准这些建议的实体。”

# 2. 获取智能体执行完批准操作后的最新会话状态（记忆）
session_end = await ner_agent_caller.get_session()

# 3. 安全获取状态字典：如果 session_end 存在则获取其 state，否则给一个空字典 {} 防报错
ner_end_state = session_end.state if session_end else {}

# 打印最终的系统状态
print("Session state: ", ner_end_state)

# 4. 最终验收测试
# 检查智能体是否成功调用了 'approve_proposed_entities' 工具，并把数据写入了 'APPROVED_ENTITIES'
if APPROVED_ENTITIES in ner_end_state:
    # 成功！打印出最终确定的实体列表。
    print("\n已批准的实体: ", ner_end_state[APPROVED_ENTITIES])
else:
    # 失败！如果人类都说了“批准”，它还没转正数据，那说明大模型跑偏了（指令遵循失败）。
    print("\n仍在等待批准？这就奇怪了。请检查智能体的状态和建议的实体。")


>>>👤 用户: 批准这些建议的实体。
<<< Agent Response: ✅ 实体类型已批准并记录为“已批准实体类型”：

- **Product**（已知）
- **Part**（已知）
- **Assembly**（已知）
- **Review**（新增）
- **Reviewer**（新增）
- **Issue**（新增）

后续的提取流程将据此从产品评论文本中识别这些命名实体，从而将评论、评论者及用户投诉问题接入现有制造 BOM 图谱，支持“用户投诉 → 部件/装配 → 供应商 → 根因”的追溯分析。

Session state:  {'approved_user_goal': {'kind_of_graph': '供应链分析', 'description': '用于制造产品的多层级物料清单（BOM），可用于根本原因分析。\n        请加入产品评论数据，以便从（用户）报告的问题（如质量、组装难度或耐用性）开始进行分析。'}, 'approved_files': ['product_reviews/gothenburg_table_reviews.md', 'product_reviews/helsingborg_dresser_reviews.md', 'product_reviews/jonkoping_coffee_table_reviews.md', 'product_reviews/linkoping_bed_reviews.md', 'product_reviews/malmo_desk_reviews.md', 'product_reviews/norrkoping_nightstand_reviews.md', 'product_reviews/orebro_lamp_reviews.md', 'product_reviews/stockholm_chair_reviews.md', 'product_reviews/uppsala_sofa_reviews.md', 'product_reviews/vasteras_bookshelf_reviews.md'], 'approved_construction_plan': {'Product': {'construction_type': 'node', 'label': 

In [19]:
## Fact Type Extraction Sub-agent

In [20]:
# 事实提取智能体 (Fact Agent) 的角色和目标
fact_agent_role_and_goal = """
  你是一个顶级的算法程序，专用于分析文本文件。
  你的目标是：根据用户的具体需求，建议可以从文本中提取哪些相关的“事实类型”（type of facts）。
"""

In [21]:
# 事实提取智能体 (Fact Agent) 的指导原则与规则
fact_agent_hints = """
  不要建议具体个别的事实（实例），而是要提出与用户目标相关的通用的“事实类型”。
  例如，不要提出“ABK喜欢咖啡”这种具体事实，而是提出通用的事实类型：“人 喜欢 饮料” (Person likes Beverage)。

  事实是由 (主语, 谓语, 宾语) —— 即 (subject, predicate, object) 组成的【三元组】。
  其中，主语（主体）和宾语（客体）必须是【已批准的实体类型】，而建议的谓语则提供它们之间如何关联的信息。
  例如，一个事实类型可以是 (Person, likes, Beverage)。

  事实的设计规则：
  - 只能使用【已批准的实体类型】作为主语或宾语。绝对不要去提出/发明新的实体类型。
  - 建议的谓语，应该准确描述已批准的主体和客体之间的关系。
  - 谓语应该针对“与用户目标高度相关的信息”进行优化（即：没用的关系不要提）。
  - 谓语必须实际出现在源文本中。不要凭空猜测（脑补/幻觉）。
  - 使用 'add_proposed_fact' 工具来记录（保存）每一个你建议的事实类型。
"""

In [22]:
# 事实提取智能体 (Fact Agent) 的思维链指导步骤
fact_agent_chain_of_thought_directions = """
    任务准备阶段：
    - 使用 'get_approved_user_goal'（获取已批准的用户目标）工具，来明确用户的核心需求。
    - 使用 'get_approved_files'（获取已批准的文件）工具，来获取需要分析的文本列表。
    - 使用 'get_approved_entities'（获取已批准的实体）工具，获取可用的实体类型列表。

    一步一步地思考（Think step by step）：
    1. 使用 'get_approved_user_goal' 工具获取用户目标（时刻牢记任务导向）。
    2. 使用 'sample_file'（抽样文件）工具，对部分授权文件进行抽样阅读，以理解文本内容。
    3. 思考文本中的主语（subjects，主体）和宾语（objects，客体）是如何关联起来的。
    4. 针对你找出的【每一个】事实类型，分别调用 'add_proposed_fact'（添加建议事实）工具进行保存。
    5. 使用 'get_proposed_facts'（获取建议事实）工具，把你刚才添加的所有事实整合提取出来。
    6. 将这些建议的事实类型展示给用户，并附上你的解释说明（为什么建议这些事实）。
"""

In [23]:
fact_agent_instruction = f"""
{fact_agent_role_and_goal}
{fact_agent_hints}
{fact_agent_chain_of_thought_directions}
"""

In [24]:
# 用于存储建议事实和已批准事实的常量（作为状态字典的键）
PROPOSED_FACTS = "proposed_fact_types" # 草稿箱：建议的事实类型
APPROVED_FACTS = "approved_fact_types" # 发布区：已批准的事实类型

def add_proposed_fact(approved_subject_label:str,
                      proposed_predicate_label:str,
                      approved_object_label:str,
                      tool_context:ToolContext) -> dict:
    """添加一个可以从文件中提取的“建议事实类型”。

    一个建议的事实类型是一个 (主语, 谓语, 宾语) 的三元组，其中
    主语和宾语必须是【已批准的实体类型】，而谓语是一个【建议的关系标签】。

    参数 (Args):
      approved_subject_label: 主语实体的已批准标签
      proposed_predicate_label: 谓语的标签（关系名）
      approved_object_label: 宾语实体的已批准标签
    """

    # 🛑 核心防御机制：防止无效/幻觉标签 (Guard against invalid labels)
    # 1. 从系统记忆中调取上一个 NER 智能体留下的【已批准实体列表】
    approved_entities = tool_context.state.get(APPROVED_ENTITIES, [])

    # 2. 检查大模型传入的主语是否合法（是否在白名单里）
    if approved_subject_label not in approved_entities:
        # 如果不合法，直接返回错误，并让大模型“重试 (Try again)”
        return tool_error(f"找不到已批准的主语标签 {approved_subject_label}。请重试。")

    # 3. 检查大模型传入的宾语是否合法
    if approved_object_label not in approved_entities:
        return tool_error(f"找不到已批准的宾语标签 {approved_object_label}。请重试。")

    # 🟢 验证通过，开始保存数据
    # 取出当前的草稿箱（如果没有就建个空字典）
    current_predicates = tool_context.state.get(PROPOSED_FACTS, {})

    # 将新的三元组存入草稿箱字典中
    current_predicates[proposed_predicate_label] = {
        "subject_label": approved_subject_label,
        "predicate_label": proposed_predicate_label,
        "object_label": approved_object_label
    }

    # 更新系统状态（记忆）
    tool_context.state[PROPOSED_FACTS] = current_predicates
    # 告诉大模型：保存成功！
    return tool_success(PROPOSED_FACTS, current_predicates)

def get_proposed_facts(tool_context:ToolContext) -> dict:
    """获取可以从文件中提取的“建议事实类型”草稿。"""
    # 将草稿箱里的事实列表提取出来（通常用于展示给人类审核）
    return tool_context.state.get(PROPOSED_FACTS, {})

def approve_proposed_facts(tool_context:ToolContext) -> dict:
    """在获得用户批准后，将建议的事实类型记录为【已批准】的事实类型。

    【注意】：仅当用户明确批准了这些事实类型后才可调用此工具。
    """
    # 错误处理：如果草稿箱是空的，报错
    if PROPOSED_FACTS not in tool_context.state:
        return tool_error("没有可批准的事实类型草稿。请先设置建议事实，请求用户批准，然后再调用此工具。")

    # 将数据从“草稿箱”移动到“正式发布区”
    tool_context.state[APPROVED_FACTS] = tool_context.state.get(PROPOSED_FACTS)
    return tool_success(APPROVED_FACTS, tool_context.state[APPROVED_FACTS])

In [25]:
fact_agent_tools = [
    get_approved_user_goal, get_approved_files,
    get_approved_entities,
    sample_file,
    add_proposed_fact,
    get_proposed_facts,
    approve_proposed_facts
]

In [26]:
# 设定 Fact 智能体的名称常量（加上 v1 版本号，方便后续升级迭代）
FACT_AGENT_NAME = "fact_type_extraction_agent_v1"

# 实例化（创建）一个智能体 (Agent) 对象
relevant_fact_agent = Agent(
    name=FACT_AGENT_NAME,
    # 智能体的名字，系统会用这个名字来称呼它

    description="提出可以从文本文件中提取的、相关事实类型的建议。",
    # 智能体的“个人简介”。如果有总管智能体（Manager Agent），总管看了这句话就知道什么时候该派它出场

    model=llm,
    # 智能体的“大脑”：指定大语言模型（比如 GPT-4o），提供推理和阅读文本的能力

    instruction=fact_agent_instruction,
    # 智能体的“灵魂/人设”：这里面打包了我们前面定义的“角色与目标(Role&Goal)”、“规则(Hints)”和“思维链(CoT)”

    tools=fact_agent_tools,
    # 智能体的“手脚”：传入我们刚刚看过的 add_proposed_fact, get_proposed_facts, approve_proposed_facts 等工具列表
)

In [27]:
# 1. 完美的交接班：Fact 智能体的初始记忆，直接复制上一个 NER 智能体的工作成果！
fact_agent_initial_state = ner_end_state.copy()

# 将 Fact 智能体和这份初始记忆绑定，生成执行器对象
fact_agent_caller = await make_agent_caller(agent=relevant_fact_agent, initial_state=fact_agent_initial_state)

# 2. 🚀 启动智能体，下达任务指令
await fact_agent_caller.chat("建议可以从文本中找到的事实类型。")

# 如果想看大模型的脑回路（详细日志），可以取消注释下一行
# await fact_agent_caller.call("Propose fact types that can be found in the text.", True)

# 获取智能体运行结束后的会话状态（此时它应该把草稿写好了）
session_end = await fact_agent_caller.get_session()

print("\n---\n")
print("\n当前会话状态: ", session_end.state)

# 打印出“已批准的实体”，确认它确实继承了上一个智能体的遗产
print("\n已批准的实体: ", session_end.state.get(APPROVED_ENTITIES, []))

# 3. 🎯 测试 1：检查能力（是否干活了？）
# 检查状态里有没有 'PROPOSED_FACTS'（草稿箱有没有东西）
if PROPOSED_FACTS in session_end.state:
    print("\n正确地建议了事实: ", session_end.state[PROPOSED_FACTS])
else:
    print("\n在状态中没找到建议的事实。哪里出问题了？")

# 4. 🛑 测试 2：检查纪律（是否听话了？）
# 检查状态里有没有 'APPROVED_FACTS'（是否偷偷把草稿转正了）
if APPROVED_FACTS in session_end.state:
    # 如果有，说明大模型越权了，必须给个差评！
    print("\n违规批准了事实: ", session_end.state[APPROVED_FACTS])
else:
    # 如果没有，说明大模型很乖，停在原地等老板审批，这才是好同志。
    print("\n没找到已批准的事实，这很好。")


>>>👤 用户: 建议可以从文本中找到的事实类型。
<<< Agent Response: 我已完成了事实类型的分析与建议。以下是基于 **10 个产品评论文件** 的抽样分析结果。

## 用户目标回顾
构建用于**供应链/根本原因分析**的图谱：以产品评论为起点，从用户报告的**质量问题、组装难度、耐用性问题**出发，追溯到具体的产品与部件。

## 建议的事实类型（共 5 个）

全部使用已批准的实体类型（Product / Part / Assembly / Review / Reviewer / Issue），且每个关系都能在评论原文中找到依据：

1. **(Review, `is_about`, Product)** — 评论针对哪个产品
   - 依据：每条评论都围绕具体产品展开，例如 *"The Gothenburg Table looks beautiful…"*、"I absolutely love my Stockholm Chair…"

2. **(Review, `written_by`, Reviewer)** — 评论的作者
   - 依据：每条评论末尾均有署名，如 *"- @scandi_lover (Minneapolis)"*。

3. **(Review, `mentions`, Issue)** — 评论中提到了哪些问题（溯源入口）
   - 依据：评论明确描述问题，如 *"The table also wobbles slightly"*、*"the drawer rails seem totally defective"*、*"the instructions were confusing"*。

4. **(Part, `has_issue`, Issue)** — 部件层面的缺陷（根本原因定位的关键）
   - 依据：大量评论把问题指向具体部件，如 *"the drawer rails… are clearly defective"*、*"the slats have started to crack"*、*"the veneer is extremely thin and already chipping"*、*"the shade which is unevenly colored"*、*"cushions 

In [28]:
# 1. 人类介入：向智能体发送明确的“批准”指令。
# 听到这句话后，Fact Agent 终于满足了调用 'approve_proposed_facts' 工具的条件。
await fact_agent_caller.chat("批准这些建议的事实类型。") #

# 2. 获取智能体执行完批准操作后的最新系统状态（记忆）
session_end = await fact_agent_caller.get_session()

# 打印最终的系统状态
print("Session state: ", session_end.state)

# 3. 最终验收测试
# 检查智能体是否成功调用了批准工具，并把数据写入了 'APPROVED_FACTS' 字典中
if APPROVED_FACTS in session_end.state:
    # 成功！打印出最终确定的事实类型（关系/连线）列表。
    print("\n已批准的事实类型: ", session_end.state[APPROVED_FACTS])
else:
    # 失败！如果你都下令“批准”了，它还没转正数据，说明指令遵循失败了。
    print("\n未能批准事实类型")


>>>👤 用户: 批准这些建议的事实类型。
<<< Agent Response: ✅ 以下 5 个事实类型已成功记录为【已批准】：

| 主语 | 谓语 | 宾语 |
|------|------|------|
| Review | `is_about` | Product |
| Review | `written_by` | Reviewer |
| Review | `mentions` | Issue |
| Part | `has_issue` | Issue |
| Product | `suffers_from` | Issue |

这组已批准的事实类型将支持从产品评论出发的供应链根本原因分析：
- **Review → Product**：定位评论针对的产品
- **Review → Reviewer**：评论来源可追溯
- **Review → Issue**：从用户评论中抓取报告的问题（质量、组装难度、耐用性）
- **Part → Issue**：将问题定位到具体部件（根本原因追溯的关键）
- **Product → Issue**：捕获成品层面的质量问题

如需进一步操作（如添加新的事实类型、分析其他文件或调整图结构），请告诉我。

Session state:  {'approved_user_goal': {'kind_of_graph': '供应链分析', 'description': '用于制造产品的多层级物料清单（BOM），可用于根本原因分析。\n        请加入产品评论数据，以便从（用户）报告的问题（如质量、组装难度或耐用性）开始进行分析。'}, 'approved_files': ['product_reviews/gothenburg_table_reviews.md', 'product_reviews/helsingborg_dresser_reviews.md', 'product_reviews/jonkoping_coffee_table_reviews.md', 'product_reviews/linkoping_bed_reviews.md', 'product_reviews/malmo_desk_reviews.md', 'product_reviews/norrkoping_nightsta